In [1]:
!pip install transformers==4.40.0 torch==2.3.0 datasets==2.19.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 104.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 99.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 83.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch
print(torch.cuda.is_available())

True


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
device= 'cuda' if torch.cuda.is_available() else 'cpu'

draft_model = AutoModelForCausalLM.from_pretrained(
    'gpt2',
    torch_dtype = torch.float16
).to(device).eval()



/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
target_model = AutoModelForCausalLM.from_pretrained(
    'gpt2-xl',
    torch_dtype = torch.float16
).to(device).eval()

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [7]:
print(torch.cuda.is_available())

True


In [8]:
from tokenizers import Tokenizer
import time
def autoregressive_generate(model, input_ids, max_new_tokens=50, temperature=1.0):
  generated = input_ids.clone()
  with torch.no_grad():
    for _ in range(max_new_tokens):
      output = model(generated)
      logits = output.logits[:, -1, :] / temperature
      probs = torch.softmax(logits, dim=-1)
      next_token = torch.multinomial(probs, num_samples=1)
      generated = torch.cat([generated, next_token], dim=-1)
      if next_token.item() == tokenizer.eos_token_id:
        break
  return generated

prompt = "Ai is the technology that will "
input_ids=tokenizer(prompt, return_tensors='pt')['input_ids'].to(device)
to = time.time()
N_RUNS = 5
for _ in range(N_RUNS):
  output = autoregressive_generate(target_model, input_ids, max_new_tokens=50)
t_baseline = (time.time()-to)/N_RUNS
print(f"Baseline Time - AVG {t_baseline:.3f}")
print(f"Tokens per sec - {50/t_baseline:.1f}")


Baseline Time - AVG 2.743
Tokens per sec - 18.2


In [16]:
def generate_draft_token(draft_model, input_ids, k=6, temperature=1.0):
  generated = input_ids.clone()
  token_output = []
  output_prob = []
  with torch.no_grad():
    for _ in range(k):
      output = draft_model(generated)
      logits = output.logits[:, -1, :] / temperature
      probs = torch.softmax(logits, dim=-1)
      next_token = torch.multinomial(probs, num_samples=1)
      generated = torch.cat([generated, next_token], dim=-1)
      token_output.append(next_token)
      output_prob.append(probs)
      #generated = torch.cat([generated, next_token], dim=-1)
      if next_token.item() == tokenizer.eos_token_id:
        break
      new_tokens = generated[:, input_ids.shape[1]:]
  return new_tokens, token_output, output_prob


k = 4
draft_seq, draft_output , draft_prob = generate_draft_token(draft_model, input_ids,k=k)
print(f"Draft Sequence - {tokenizer.decode(draft_seq[0])}")
#print(f"Draft Output - {tokenizer.decode(draft_output[0])}")
print(f"Draft Prob Shape - {draft_prob[0].shape}")


Draft Sequence -  develop tomorrow's
Draft Prob Shape - torch.Size([1, 50257])


In [18]:
def verify_draft_tokens(target_model, input_ids, draft_sequence, temperature=1.0):
    K = draft_sequence.shape[1]
    full_sequence = torch.cat([input_ids, draft_sequence ],dim=-1)
    #finding the output of the target model
    with torch.no_grad():
        output = target_model(full_sequence)
        logits = output.logits

    if temperature != 1:
        logits = logits/temperature


    target_probs = []
    sequence_length  = input_ids.shape[1]

    for i in range(K):
        #this eq is basically points to the position of the next token, as LLM at pos n predicts the words n+1
        pos = sequence_length-1+i
        probs = torch.softmax(logits[:,pos,:],dim=-1)
        target_probs.append(probs)
        draft_token = draft_sequence[0, i]

        draft_prob = probs[0, draft_token]

        print("Draft token:",
            tokenizer.decode([draft_token]))

        print("Target probability:",
            draft_prob.item())

    bonus_probs = torch.softmax(logits[:, -1, :], dim=-1)

    return target_probs,bonus_probs



target_probs, bonus_probs = verify_draft_tokens(
target_model, input_ids, draft_seq)
print(f'Got {len(target_probs)} target distributions')
print(f'Each shape: {target_probs[0].shape}')


Draft token:  
Target probability: 0.5302734375
Draft token: develop
Target probability: 0.0015897750854492188
Draft token:  tomorrow
Target probability: 0.0015058517456054688
Draft token: 's
Target probability: 0.77978515625
Got 4 target distributions
Each shape: torch.Size([1, 50257])


In [34]:
def rejection_sampling(draft_token, draft_prob,target_probs,bonus_probs):


    K = draft_token.shape[1]
    accepted = []
    n_accepted = 0  

    print(f"draft token: {draft_token}")
    #print(tokenizer.decode(draft_token))
    print(f"draft prob: {draft_prob}")

    for i in range(k):
        #taking 1 token at a time, token.item() converts it to INT.
        token = draft_token[:, i:i+1]
        token_id = token.item()


        #finding the draft and target prob of the token.
        p = draft_prob[i][0,token_id].item()
        q = target_probs[i][0,token_id].item()


        #if the acceptance_rate is 1, we accept the token, if its less than 1, then we need to compare it to a randn num. 
        acceptance =  min(1,q/p+1e-10)
        num = torch.rand(1).item()
        print("Draft token:",
        tokenizer.decode(token.squeeze()))

        print("Draft prob:", p)

        print("Target prob:", q)

        print("Acceptance:", acceptance)

        print("Random num:", num)
        print("-----")
        if num<= acceptance:
            accepted.append(token)
            n_accepted+=1
        else:
                        # REJECT: resample from corrected distribution
            corrected = torch.clamp(
            target_probs[i] - draft_prob[i], min=0)
            corrected = corrected / corrected.sum() # normalize
            resampled = torch.multinomial(corrected, num_samples=1)
            accepted.append(resampled)
            # Stop — all subsequent tokens invalid
            break
        if n_accepted == K:
            bonus_token = torch.multinomial(bonus_probs, num_samples=1)
            accepted.append(bonus_token)
    return torch.cat(accepted, dim=-1), n_accepted
# Test
accepted, n = rejection_sampling(draft_seq, draft_prob, target_probs, bonus_probs)
#print(f'Accepted {n}/{K} draft tokens')
print("Accepted sequence:")
print(tokenizer.decode(accepted.squeeze()))

draft token: tensor([[ 1849, 16244,  9439,   338]], device='cuda:0')
draft prob: [tensor([[1.0443e-04, 3.8385e-05, 5.9009e-06,  ..., 0.0000e+00, 0.0000e+00,
         1.9312e-05]], device='cuda:0', dtype=torch.float16), tensor([[9.4771e-06, 3.3426e-04, 8.3447e-06,  ..., 0.0000e+00, 0.0000e+00,
         6.5565e-07]], device='cuda:0', dtype=torch.float16), tensor([[1.2577e-04, 4.6730e-04, 2.8014e-06,  ..., 1.7881e-07, 1.1921e-07,
         7.1645e-05]], device='cuda:0', dtype=torch.float16), tensor([[6.8235e-04, 7.7343e-04, 4.2915e-06,  ..., 0.0000e+00, 0.0000e+00,
         8.6784e-05]], device='cuda:0', dtype=torch.float16)]
Draft token:  
Draft prob: 0.43896484375
Target prob: 0.5302734375
Acceptance: 1
Random num: 0.7804374098777771
-----
Draft token: develop
Draft prob: 0.0067138671875
Target prob: 0.0015897750854492188
Acceptance: 0.23678977282727273
Random num: 0.8278917074203491
-----
Accepted sequence:
 be
